# Reorder Point Analysis

Small shop owners often use a single fixed threshold ("reorder when stock hits 5") to decide when to restock, regardless of how fast a product actually sells. This notebook builds and tests a better rule: a reorder point based on each product's own average daily sales and how long its supplier takes to deliver.

This is the same logic already running live in InsightDuka's `js/db.js` file, prototyped here first so it can be explained, tested, and compared against a naive baseline.

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

dates = pd.date_range(end=pd.Timestamp.today(), periods=30)

products = ['Unga wa Dola 2kg', 'Sukari 1kg', 'Mafuta Elianto 1L']
avg_daily_sales = {'Unga wa Dola 2kg': 1, 'Sukari 1kg': 3, 'Mafuta Elianto 1L': 0.5}

rows = []
for product in products:
    for date in dates:
        units = np.random.poisson(avg_daily_sales[product])
        rows.append({'date': date, 'product': product, 'units_sold': units})

sales = pd.DataFrame(rows)
sales.head(10)

,date,product,units_sold
0,2026-09-03 00:09:24.003312,Unga wa Dola 2kg,1
1,2026-09-04 00:09:24.003312,Unga wa Dola 2kg,2
2,2026-09-05 00:09:24.003312,Unga wa Dola 2kg,0
3,2026-09-06 00:09:24.003312,Unga wa Dola 2kg,0
4,2026-09-07 00:09:24.003312,Unga wa Dola 2kg,3
5,2026-09-08 00:09:24.003312,Unga wa Dola 2kg,2
6,2026-09-09 00:09:24.003312,Unga wa Dola 2kg,0
7,2026-09-10 00:09:24.003312,Unga wa Dola 2kg,0
8,2026-09-11 00:09:24.003312,Unga wa Dola 2kg,0
9,2026-09-12 00:09:24.003312,Unga wa Dola 2kg,1


## Generating realistic sales data

Real sales don't come in neat, constant amounts, a product averaging 1 unit a day might sell 0 on a quiet day and 3 on a busy one. The data above uses a Poisson distribution, a standard way to model "how many events happen per day" when you know the average rate, to generate 30 days of sales for three products with different typical speeds.

In [3]:
actual_averages = sales.groupby('product')['units_sold'].mean()
actual_averages

product
Mafuta Elianto 1L    0.500000
Sukari 1kg           2.766667
Unga wa Dola 2kg     0.866667
Name: units_sold, dtype: float64

## The naive approach: a fixed threshold

A common shortcut is "reorder when stock drops to some fixed number," say, 5 units, regardless of the product. Let's see where that breaks down using the averages above.

In [4]:
lead_time_days = 4

for product in products:
    daily_avg = actual_averages[product]
    expected_demand_during_wait = daily_avg * lead_time_days
    print(f"{product}: sells {daily_avg:.2f}/day, expected demand during {lead_time_days}-day wait = {expected_demand_during_wait:.2f}")

Unga wa Dola 2kg: sells 0.87/day, expected demand during 4-day wait = 3.47
Sukari 1kg: sells 2.77/day, expected demand during 4-day wait = 11.07
Mafuta Elianto 1L: sells 0.50/day, expected demand during 4-day wait = 2.00


## Why the fixed threshold fails

A fixed reorder point of 5 units works fine for slow sellers, but not for fast ones. The table below compares the naive threshold against actual expected demand during the supplier wait, for each product.

In [5]:
naive_threshold = 5

comparison = pd.DataFrame({
    'product': products,
    'avg_daily_sales': [actual_averages[p] for p in products],
    'expected_demand_during_wait': [actual_averages[p] * lead_time_days for p in products],
    'naive_threshold': naive_threshold
})

comparison['naive_threshold_enough'] = comparison['expected_demand_during_wait'] <= comparison['naive_threshold']
comparison

,product,avg_daily_sales,expected_demand_during_wait,naive_threshold,naive_threshold_enough
0,Unga wa Dola 2kg,0.866667,3.466667,5,True
1,Sukari 1kg,2.766667,11.066667,5,False
2,Mafuta Elianto 1L,0.500000,2.000000,5,True


## The actual reorder point formula

Instead of one fixed number for every product, the reorder point should scale with how fast that specific product sells: average daily sales × supplier lead time, plus a safety buffer for days that sell faster than average.

In [6]:
safety_stock = 3

comparison['reorder_point'] = comparison['avg_daily_sales'] * lead_time_days + safety_stock
comparison['reorder_point'] = comparison['reorder_point'].apply(lambda x: int(np.ceil(x)))
comparison

,product,avg_daily_sales,expected_demand_during_wait,naive_threshold,naive_threshold_enough,reorder_point
0,Unga wa Dola 2kg,0.866667,3.466667,5,True,7
1,Sukari 1kg,2.766667,11.066667,5,False,15
2,Mafuta Elianto 1L,0.500000,2.000000,5,True,5


## Conclusion

The naive fixed threshold of 5 units only happened to work for the slower-selling products. For Sukari, which sells nearly three times faster than Unga, the real reorder point is 15 units, not 5, meaning a shop owner using the fixed rule would run out of stock roughly a week before the naive alert ever fired.

This reorder point formula is already running live in InsightDuka (`js/db.js`), computed per product from each product's own 14-day sales history.